In this we are creating a multilayer perceptron for making trigrams.

-- the aim is to consider 3 words / characters as context and predict the next word/character.
-- for this we are going to create a multilayer perceptron with input layer, then have an hidden layer as an hyperparameter and then finally have an softmax layer to get the probablity distribution to get the next best word.
-- the paper we are refering to consist of an example where 3 words are as context to predict the next word. 
-- and we convert each word to 1x30 that is 30 dimension vector.

In [1]:
## making the basic imports
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
## Load the words as in the previous example
words = open("names.txt", 'r').read().splitlines()

## Generate stoi and itos mappings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}

print(stoi)
print(itos)


{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
## lets generate the data set with trigrams:
X, Y = [], []
block_size = 3
for w in words[:8]:
    context = [0] * block_size ## Since we are operating at character level we create a new block / context window for each word.
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} -----> {itos[ix]}")
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

... -----> e
..e -----> m
.em -----> m
emm -----> a
mma -----> .
... -----> o
..o -----> l
.ol -----> i
oli -----> v
liv -----> i
ivi -----> a
via -----> .
... -----> a
..a -----> v
.av -----> a
ava -----> .
... -----> i
..i -----> s
.is -----> a
isa -----> b
sab -----> e
abe -----> l
bel -----> l
ell -----> a
lla -----> .
... -----> s
..s -----> o
.so -----> p
sop -----> h
oph -----> i
phi -----> a
hia -----> .
... -----> c
..c -----> h
.ch -----> a
cha -----> r
har -----> l
arl -----> o
rlo -----> t
lot -----> t
ott -----> e
tte -----> .
... -----> m
..m -----> i
.mi -----> a
mia -----> .
... -----> a
..a -----> m
.am -----> e
ame -----> l
mel -----> i
eli -----> a
lia -----> .


In [12]:
## Current dimesions of the dataset

X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([53, 3]), torch.int64, torch.Size([53]), torch.int64)

In [15]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  3],
        [ 0,  3,  8],
        [ 3,  8,  1],
        [ 8,  1, 18],
        [ 1, 18, 12],
        [18, 12, 15],
        [12, 15, 20],
        [15, 20, 20],
        [20, 20,  5],
        [ 0,  0,  0],
        [ 0,  0, 13],
        [ 0, 13,  9],
        [1

In [21]:
## Lets embed things

C = torch.randn((27,2)) ## This is analogous to 17000 x 30 ---> 27 x 2

## So basically now we have a C array that is 27x2 that sets a 2 dim value for each character in the 27 characters 
## Additionally we we say C[X] we are already havein X of nx3 so for each character in word of X --> X[n[m]] the C assigns a 2 dimensional vector is assigned making it C[X] = nx3x2

print(X.shape, C.shape, C[X].shape)
print(C[X][0]) ## which is the first context of the first character we predicted with 2 dim embedding for each such char. 
print("-----------------")
print(C[X])

torch.Size([53, 3]) torch.Size([27, 2]) torch.Size([53, 3, 2])
tensor([[ 0.6362, -0.9603],
        [ 0.6362, -0.9603],
        [ 0.6362, -0.9603]])
-----------------
tensor([[[ 0.6362, -0.9603],
         [ 0.6362, -0.9603],
         [ 0.6362, -0.9603]],

        [[ 0.6362, -0.9603],
         [ 0.6362, -0.9603],
         [ 0.2200, -0.8161]],

        [[ 0.6362, -0.9603],
         [ 0.2200, -0.8161],
         [-0.5847,  0.4981]],

        [[ 0.2200, -0.8161],
         [-0.5847,  0.4981],
         [-0.5847,  0.4981]],

        [[-0.5847,  0.4981],
         [-0.5847,  0.4981],
         [-0.2254, -1.1717]],

        [[ 0.6362, -0.9603],
         [ 0.6362, -0.9603],
         [ 0.6362, -0.9603]],

        [[ 0.6362, -0.9603],
         [ 0.6362, -0.9603],
         [-0.0378,  1.2605]],

        [[ 0.6362, -0.9603],
         [-0.0378,  1.2605],
         [-1.2312,  1.7086]],

        [[-0.0378,  1.2605],
         [-1.2312,  1.7086],
         [-1.1059, -1.4689]],

        [[-1.2312,  1.7086],
    